In [137]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

True

In [138]:
client = OpenAI(
    api_key=os.environ['api_key'],
    base_url=os.environ['base_url'],
)

In [139]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "查询指定城市的天气情况，一次只能查询一个城市",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "要查询的城市名称，例如：北京",
                    }
                },
                "required": ["city"]
            },
        }
    },
    {
        "type": "function",
        "function": {
            "name": "send_msg",
            "description": "发送天气提醒给用户，只要含有通知字样的，调用这个工具",
            "parameters": {
                "type": "object",
                "properties": {
                    "message": {
                        "type": "string",
                        "description": "发送天气提醒给用户，只要含有通知字样的，调用这个工具",
                    }
                },
                "required": ["message"]
            },
        }
    },
]

In [140]:
def get_weather(city):
    return {
        "城市": city,
        "天气": "晴",
        "温度": "25°C",
        "湿度": "60%",
        "风速": "5 km/h"
    }
def send_msg(message):
    """
    发送天气提醒给用户
    """
    return {
        "status": "success",
        "message": f"已发送消息: {message}"
    }

In [141]:
available_functions = {
    "get_weather": get_weather,
    "send_msg": send_msg
}

In [142]:
messages = [
    {"role": "user", "content": "请帮我查询北京和上海的天气情况，并将天气结果通知给makuolang"}
]

response = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=messages,
    tools=tools
)

In [143]:
#模型第一次返回
response.choices[0].message.tool_calls

[ChatCompletionMessageFunctionToolCall(id='call_00_kbfC4bkEMkjJqcQiNB5d2736', function=Function(arguments='{"city": "北京"}', name='get_weather'), type='function', index=0),
 ChatCompletionMessageFunctionToolCall(id='call_01_laQq5buhTq0BZsRShuAq5894', function=Function(arguments='{"city": "上海"}', name='get_weather'), type='function', index=1)]

In [144]:
import json
def exec_tool(messages, response):

    messages.append(response.choices[0].message.model_dump())
    #要调用的函数
    tool_calls = response.choices[0].message.tool_calls

    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments)
        function_to_call = available_functions[tool_name]  # 获取到对应的工具函数

        function_response = function_to_call(**tool_args)

        messages.append({
            "role": "tool",   # 固定写法，表示这是工具调用的结果
            "tool_call_id": tool_call.id,   # 关联到具体的工具调用id
            "content": str(function_response)  # 工具执行的结果内容，必须是字符串格式
        })
    return messages

In [145]:
messages = exec_tool(messages, response)
messages

[{'role': 'user', 'content': '请帮我查询北京和上海的天气情况，并将天气结果通知给makuolang'},
 {'content': '我来为您查询北京和上海的天气情况。',
  'refusal': None,
  'role': 'assistant',
  'annotations': None,
  'audio': None,
  'function_call': None,
  'tool_calls': [{'id': 'call_00_kbfC4bkEMkjJqcQiNB5d2736',
    'function': {'arguments': '{"city": "北京"}', 'name': 'get_weather'},
    'type': 'function',
    'index': 0},
   {'id': 'call_01_laQq5buhTq0BZsRShuAq5894',
    'function': {'arguments': '{"city": "上海"}', 'name': 'get_weather'},
    'type': 'function',
    'index': 1}],
  'reasoning_content': 'The user wants to query Beijing and Shanghai weather, then notify makuolang. Let me query both cities - but the tool says "一次只能查询一个城市" (only one city at a time). So I need to make two separate calls. I can make them in the same block since they\'re independent.\n\nThen I need to send a notification message.\n\nLet me first query both.'},
 {'role': 'tool',
  'tool_call_id': 'call_00_kbfC4bkEMkjJqcQiNB5d2736',
  'content': "{'城市': '

In [147]:
second_response = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=messages,
    tools=tools
)
second_response.choices[0].finish_reason

'tool_calls'

In [148]:
third_response  = exec_tool(messages, second_response)

result = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=third_response,
    tools=tools
)

result.choices[0].message.content

'已为您完成查询并通知，结果如下：\n\n**北京**：晴 ☀️，气温 25°C，湿度 60%，风速 5 km/h\n**上海**：晴 ☀️，气温 25°C，湿度 60%，风速 5 km/h\n\n天气提醒已成功发送给 makuolang ✅\n\n两地天气都不错，适合外出活动，注意适当补水和防晒。'

In [151]:
from IPython.display import Markdown
display(Markdown(result.choices[0].message.content))

已为您完成查询并通知，结果如下：

**北京**：晴 ☀️，气温 25°C，湿度 60%，风速 5 km/h
**上海**：晴 ☀️，气温 25°C，湿度 60%，风速 5 km/h

天气提醒已成功发送给 makuolang ✅

两地天气都不错，适合外出活动，注意适当补水和防晒。